# Global descriptors generation

This notebook can be used to generate a set of global descriptors using the MegaLoc model and test retrival in the second part

**Note**: create and fill a `.env` file at the root of the project with the same structure as `template.env`

In [1]:
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from dotenv import load_dotenv
import os
import numpy as np

from safetensors.torch import load_file
from MegaLoc.megaloc_model import MegaLoc


## Global variables

In [3]:
load_dotenv()
IMAGES_FOLDER = Path(os.getenv("IMAGES_FOLDER", ""),)
DESCRIPTORS_FILE   = Path(os.getenv("IMAGES_DESCRIPTORS", ""))
MEGALOC_WEIGHTS_PATH = Path(os.getenv("MEGALOC_WEIGHTS_PATH", ""))
TEST_IMAGES_FOLDER = Path(os.getenv("TEST_IMAGES_FOLDER", ""))

IMG_SIZE     = 322
BATCH_SIZE   = 32
NUM_WORKERS  = 4
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"


## Model Loading

In [5]:
def get_trained_model(weights_path) -> torch.nn.Module:
    """Load the pretrained MegaLoc model.

    Returns:
        MegaLoc model with pretrained weights.
    """
    model = MegaLoc()

    state_dict = load_file(weights_path)
    model.load_state_dict(state_dict)

    return model

In [6]:
print("Device:", DEVICE)
model = get_trained_model(MEGALOC_WEIGHTS_PATH)
model.eval().to(DEVICE)


Device: cuda


MegaLoc(
  (backbone): DINOv2(
    (patch_embed): PatchEmbedding(
      (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    )
    (blocks): ModuleList(
      (0-11): 12 x TransformerBlock(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
        (ls2): LayerScale()
      )
    )
    (norm): LayerNorm((768,),

## Data preparation

In [7]:
# Same process as in https://github.com/serizba/salad/blob/main/dataloaders/GSVCitiesDataloader.py
preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

class PosedCameraImages(Dataset):
    def __init__(self, root, transform):
        root = Path(root)
        self.paths = sorted(p for p in root.glob("*") if p.suffix.lower() in [".png"])
        if not self.paths:
            sys.exit(f"No images found under {root}")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), idx

dataset = PosedCameraImages(IMAGES_FOLDER, preprocess)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Found {len(dataset)} images.")


Found 11911 images.


## Global descriptors generation

In [ ]:
if DESCRIPTORS_FILE.exists():
    # delete the file if exists
    print(f"Warning: {DESCRIPTORS_FILE} already exists.")
    choice = input("Continue with (y)es or stop with n(o): ")
    if choice.lower() != "y":
        sys.exit(0)
    os.remove(DESCRIPTORS_FILE)

descriptors_list = []
with torch.no_grad():
    for images, _ in tqdm(loader, desc="Extraction"):
        images = images.to(DEVICE, non_blocking=True)
        descriptor = model(images)
        descriptors_list.append(descriptor.cpu().numpy().astype(np.float32))

descriptors_concat = np.concatenate(descriptors_list, axis=0)
np.savez(
    DESCRIPTORS_FILE, descs=descriptors_concat, paths=np.array([str(p) for p in dataset.paths])
)
print(f"Saved descriptors {descriptors_concat.shape} to {DESCRIPTORS_FILE}")

# list of paths, aligned with descriptors_concat rows
paths = dataset.paths


## Retrieval results

In [8]:
def retrieve_from_path(image_path, top_k=5):
    """Encode an external image and search against the indexed database."""
    query_image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        query_descriptor = F.normalize(model(query_image), p=2, dim=1)
    similarity_scores = (query_descriptor @ descriptors_torch.T).squeeze(0)
    scores, indexes = torch.topk(similarity_scores, k=top_k)
    return indexes.cpu().numpy(), scores.cpu().numpy()


def show_external_retrieval(query_path, top_k=5):
    indexes, scores = retrieve_from_path(query_path, top_k=top_k)
    fig, axes = plt.subplots(1, top_k + 1, figsize=(3 * (top_k + 1), 3.2))

    axes[0].imshow(Image.open(query_path).convert("RGB"))
    axes[0].set_title(f"QUERY\n{Path(query_path).name}", fontsize=9)
    axes[0].axis("off")

    for ax, i, s in zip(axes[1:], indexes, scores):
        ax.imshow(Image.open(paths[i]).convert("RGB"))
        ax.set_title(f"#{i}  sim={s:.3f}\n{paths[i].name}", fontsize=8)
        ax.axis("off")

    plt.tight_layout()

    output_dir = Path("./data_exploration_outputs")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"retrival_{Path(query_path).stem}.pdf"
    fig.savefig(output_path, format="pdf", bbox_inches="tight")
    plt.close(fig)


**Note:** commented code, just to preprocess the images

In [4]:
# for p in Path(TEST_IMAGES_FOLDER).glob("*.png"):
#     img = Image.open(p)
#     w, h = img.size
#     img.resize((round(w * 720 / h), 720), Image.LANCZOS).save(p)

from heic2png import HEIC2PNG

for p in Path(TEST_IMAGES_FOLDER).glob("*.HEIC"):
    img = HEIC2PNG(p).image
    w, h = img.size
    img.resize((round(w * 720 / h), 720), Image.LANCZOS).save(
        p.with_suffix(".png"), format="PNG"
    )


In [9]:
descriptors_files = np.load(DESCRIPTORS_FILE, allow_pickle=True)
descriptors_concat = descriptors_files["descs"]
paths = [Path(p) for p in descriptors_files["paths"]]
print(f"Loaded descriptors {descriptors_concat.shape} from {DESCRIPTORS_FILE}")

test_paths = sorted(p for p in Path(TEST_IMAGES_FOLDER).glob("*") if p.suffix.lower() in [".png"])
print(f"Found {len(test_paths)} test images.")
descriptors_torch = torch.from_numpy(descriptors_concat).to(DEVICE)


Loaded descriptors (11911, 8448) from /ronflex/users/sgar@inno.evs.tv/datasets/montefiore/video_1_frames/megaloc_descriptors.npz
Found 47 test images.


In [10]:
for query_image_path in test_paths:
    show_external_retrieval(query_image_path, top_k=5)


#### Test dataset description

- 21 images extracted from a video (iPhone 13 Mini), conditions: some blury images, different camera than the one used for the database images, recorded at night
- 26 images taken "manually" (iPhone 13 Mini), conditions: different camera than the one used for the database images, recorded in day (same as Database images), some (13 images) taken with a human in it (to simulate "real" busy corridor).

#### Results

Precision@1 (4m): 45/47 (= 95.74%)
- Supports that MegaLoc can be a viable solution for our VPR problem.

Precision@5 (4m): TODO
- Supports that the dataset is relatievely rich since it contains at least 5 correct images for a query in most cases.
